# 02 – Iteration 3: Feature Engineering for Dataset 2

This notebook:

1. Loads the **cleaned Dataset 2** produced in `01_iter3_data_cleaning.ipynb`.
2. Builds a **row-level feature table** (per reading, per meter context) using `src/features.py`:
   - Lags & deltas: `cons_lag1`, `cons_lag2`, `delta1`, `delta2`
   - Rolling statistics on last N readings (N ∈ {3, 12, 24}):
     mean, std, min, max, zero_ratio, neg_ratio
   - Meter-level z-score vs. historical mean/std: `cons_z_meter`
   - Period duration in hours: `period_hours` (if `data_inici` & `data_fi` exist)
3. Adds the binary label **`y_anom`** derived from the anomaly code column.
4. Saves the feature table in:
   - `iteration_3/results/features/features_dataset2_iter3.parquet`

This feature table will be used in:

- `03_iter3_feature_selection.ipynb`
- `04_iter3_dataset_preparation.ipynb`


Imports & paths

In [1]:
import os
import sys

import pandas as pd

# Make sure we can import from src/
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from features import build_features

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

CLEANED_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "cleaned"))
FEATURES_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "features"))

os.makedirs(FEATURES_DIR, exist_ok=True)

CLEANED_BASE_NAME = "dataset2_cleaned_iter3"
CLEANED_PARQUET_PATH = os.path.join(CLEANED_DIR, f"{CLEANED_BASE_NAME}.parquet")

FEATURES_PARQUET_PATH = os.path.join(FEATURES_DIR, "features_dataset2_iter3.parquet")

print("PROJECT_ROOT        :", PROJECT_ROOT)
print("CLEANED_DIR         :", CLEANED_DIR)
print("FEATURES_DIR        :", FEATURES_DIR)
print("CLEANED_PARQUET_PATH:", CLEANED_PARQUET_PATH)
print("FEATURES_PARQUET_PATH:", FEATURES_PARQUET_PATH)

if not os.path.exists(CLEANED_PARQUET_PATH):
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_PARQUET_PATH}")


PROJECT_ROOT        : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D
CLEANED_DIR         : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\cleaned
FEATURES_DIR        : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\features
CLEANED_PARQUET_PATH: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\cleaned\dataset2_cleaned_iter3.parquet
FEATURES_PARQUET_PATH: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\features\features_dataset2_iter3.parquet


Load cleaned dataset

In [2]:
# 1) Load cleaned Dataset 2
df_clean = pd.read_parquet(CLEANED_PARQUET_PATH)

print("Cleaned dataset loaded.")
print("Shape:", df_clean.shape)
display(df_clean.head())


Cleaned dataset loaded.
Shape: (21195970, 11)


,POLISSA_SUBM,CODI_ANOMALIA,START_DATE,END_DATE,US_AIGUA_SUBM,SECCIO_CENSAL,NUMEROSERIECONTADOR,CONSUMO_REAL,FECHA_HORA,flag_anom_32768,flag_anom_163840
0,TZSHLTAPLXX4OYI3,163840,2024-07-08,2024-09-05,DOMÈSTIC,0805601006,P22FA037836K,NaN,2024-01-01,False,True
1,FJ2I5K246X6SG3T4,163840,2023-01-26,2023-03-27,DOMÈSTIC,0805602004,P21VA155772I,NaN,2024-01-01,False,True
2,MPIXKKMZKJXANKKB,163840,2024-05-07,2024-07-05,DOMÈSTIC,0801507024,I20LA206734D,NaN,2024-01-01,False,True
3,LV6FI7TE7BX7NKKE,163840,2023-01-11,2023-03-13,DOMÈSTIC,0820002005,I19LA121835K,NaN,2024-01-01,False,True
4,RSSOFEQOC53RL6OD,2,2023-01-16,2023-03-16,DOMÈSTIC,0801906059,P15VA076725J,NaN,2024-01-01,False,False


Build features (using build_features)

In [3]:
df_clean.columns.tolist()


['POLISSA_SUBM',
 'CODI_ANOMALIA',
 'START_DATE',
 'END_DATE',
 'US_AIGUA_SUBM',
 'SECCIO_CENSAL',
 'NUMEROSERIECONTADOR',
 'CONSUMO_REAL',
 'FECHA_HORA',
 'flag_anom_32768',
 'flag_anom_163840']

In [ ]:
df_clean.shape

(21195970, 11)

In [ ]:
# 2) Build row-level features (lags, rolling stats, meter stats, period hours, y_anom)

feat_df = build_features(df_clean)

print("Feature table created.")
print("Feature shape:", feat_df.shape)
display(feat_df.head())


Feature table created.
Feature shape: (21195970, 19)


,POLISSA_SUBM,CODI_ANOMALIA,START_DATE,END_DATE,US_AIGUA_SUBM,SECCIO_CENSAL,NUMEROSERIECONTADOR,CONSUMO_REAL,FECHA_HORA,flag_anom_32768,flag_anom_163840,y_anom,datetime,cons_lag1,delta1,meter_mean,meter_std,cons_z_meter,period_hours
491,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 00:36:20,False,True,1,2024-01-01 00:36:20,NaN,NaN,0.0,0.0,NaN,1488.0
1190,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 01:36:20,False,True,1,2024-01-01 01:36:20,0.0,0.0,0.0,0.0,NaN,1488.0
1892,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 02:36:32,False,True,1,2024-01-01 02:36:32,0.0,0.0,0.0,0.0,NaN,1488.0
2588,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 03:36:32,False,True,1,2024-01-01 03:36:32,0.0,0.0,0.0,0.0,NaN,1488.0
3286,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 04:36:32,False,True,1,2024-01-01 04:36:32,0.0,0.0,0.0,0.0,NaN,1488.0


In [9]:
display(feat_df.head(-20))

,POLISSA_SUBM,CODI_ANOMALIA,START_DATE,END_DATE,US_AIGUA_SUBM,SECCIO_CENSAL,NUMEROSERIECONTADOR,CONSUMO_REAL,FECHA_HORA,flag_anom_32768,flag_anom_163840,y_anom,datetime,cons_lag1,delta1,meter_mean,meter_std,cons_z_meter,period_hours
491,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 00:36:20,False,True,1,2024-01-01 00:36:20,NaN,NaN,0.000000,0.000000,NaN,1488.0
1190,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 01:36:20,False,True,1,2024-01-01 01:36:20,0.0,0.0,0.000000,0.000000,NaN,1488.0
1892,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 02:36:32,False,True,1,2024-01-01 02:36:32,0.0,0.0,0.000000,0.000000,NaN,1488.0
2588,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 03:36:32,False,True,1,2024-01-01 03:36:32,0.0,0.0,0.000000,0.000000,NaN,1488.0
3286,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 04:36:32,False,True,1,2024-01-01 04:36:32,0.0,0.0,0.000000,0.000000,NaN,1488.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20763769,XYTFK6XEPHX75PGL,262144,2023-07-26,2023-09-27,DOMÈSTIC,0801905065,P24VA137717A,15.0,2024-12-31 11:13:47,False,False,1,2024-12-31 11:13:47,9.0,6.0,6.748049,20.871713,0.395365,1512.0
20764639,XYTFK6XEPHX75PGL,262144,2023-07-26,2023-09-27,DOMÈSTIC,0801905065,P24VA137717A,54.0,2024-12-31 12:13:47,False,False,1,2024-12-31 12:13:47,15.0,39.0,6.748049,20.871713,2.263923,1512.0
20765508,XYTFK6XEPHX75PGL,262144,2023-07-26,2023-09-27,DOMÈSTIC,0801905065,P24VA137717A,0.0,2024-12-31 13:13:47,False,False,1,2024-12-31 13:13:47,54.0,-54.0,6.748049,20.871713,-0.323311,1512.0
20766379,XYTFK6XEPHX75PGL,262144,2023-07-26,2023-09-27,DOMÈSTIC,0801905065,P24VA137717A,0.0,2024-12-31 14:13:47,False,False,1,2024-12-31 14:13:47,0.0,0.0,6.748049,20.871713,-0.323311,1512.0


Quick diagnostics (NA, zero-variance, label distribution)

In [6]:
# 3) Quick diagnostics for documentation

# Label distribution
if "y_anom" in feat_df.columns:
    print("y_anom counts:", feat_df["y_anom"].value_counts(dropna=False).to_dict())
else:
    print("[warn] 'y_anom' not found in feature table.")

# Numeric columns
num_cols = [c for c in feat_df.columns if pd.api.types.is_numeric_dtype(feat_df[c])]
print(f"Numeric feature count: {len(num_cols)}")

# Zero-variance numeric features
zero_var = [c for c in num_cols if feat_df[c].std(skipna=True) == 0]
print(f"Zero-variance numeric features: {len(zero_var)}")
if zero_var:
    print("Example zero-variance features:", zero_var[:20])

# NA burden across numeric features (top 15)
na_burden = (
    feat_df[num_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(15)
)
print("Top-15 NA ratios among numeric features:")
display(na_burden)


y_anom counts: {1: 21185222, 0: 10748}
Numeric feature count: 11
Zero-variance numeric features: 0
Top-15 NA ratios among numeric features:


cons_z_meter        0.243938
delta1              0.235251
cons_lag1           0.211815
CONSUMO_REAL        0.211699
meter_mean          0.001789
meter_std           0.001789
CODI_ANOMALIA       0.000000
flag_anom_32768     0.000000
y_anom              0.000000
flag_anom_163840    0.000000
period_hours        0.000000
dtype: float64

Save features table

In [7]:
# 4) Save feature table for later steps

feat_df.to_parquet(FEATURES_PARQUET_PATH, index=False)

print(f"[ok] Saved features to: {FEATURES_PARQUET_PATH}")


[ok] Saved features to: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\features\features_dataset2_iter3.parquet
